# TakeMeter — Fine-Tuning Starter Notebook
### AI201 · Project 3

This notebook walks you through fine-tuning a text classifier on your annotated dataset and comparing it to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Runs the fine-tuning pipeline with DistilBERT
- Computes evaluation metrics and generates a confusion matrix
- Runs the Groq baseline and compares both models

**What you do (the actual work):**
- Collect and annotate your 200+ examples (done before opening this notebook)
- Define your label map and upload your CSV
- Write your Groq classification prompt using your label definitions
- Analyze the output and write your evaluation report

---
**Before you start:** Make sure you are using a T4 GPU runtime.  
Go to **Runtime → Change runtime type → T4 GPU**, then click Save.

In [1]:
# Install any dependencies not pre-installed on Colab
!pip install -q groq python-dotenv
print("✅ Dependencies ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.8 MB/s eta 0:00:00
✅ Dependencies ready


In [ ]:
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map.  
Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).

In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────
# Define YOUR label map below.
# Keys are the string labels in your CSV; values are integers starting at 0.
# ────────────────────────────────────────────────────────────────────────

LABEL_MAP = {
    "due_diligence": 0,
    "hot_take": 1,
    "reaction": 2,
}

# ── END TODO ──────────────────────────────────────────────────────────────

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
print(f"Labels: {LABEL_MAP}")
print(f"Number of labels: {NUM_LABELS}")

In [ ]:
# Upload your CSV from your computer
from google.colab import files
print("Select your labeled dataset CSV file...")
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {CSV_PATH}")

In [ ]:
# Load and validate your dataset
df = pd.read_csv(CSV_PATH)

# ── TODO (if needed) ──────────────────────────────────────────────────────
# If your CSV uses different column names, rename them here.
# Example: df = df.rename(columns={"post": "text", "category": "label"})
# ── END TODO ──────────────────────────────────────────────────────────────

print(f"Columns: {df.columns.tolist()}")
print(f"Total examples: {len(df)}")
print()
print("Label distribution:")
print(df["label"].value_counts())

# Validate all labels are in LABEL_MAP
unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    print(f"\n⚠️  Labels in CSV not found in LABEL_MAP: {unknown}")
    print("Update your LABEL_MAP above to include all labels.")
else:
    print("\n✅ All labels match your LABEL_MAP")

# Convert string labels to integers
df["label_id"] = df["label"].map(LABEL_MAP)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

---
## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.

In [ ]:
# Train / val / test split — 70% / 15% / 15%
# Stratified so each split has roughly the same label distribution.
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"]
)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print()
print("Train label distribution:")
print(train_df["label"].value_counts())
print()
print("Test label distribution:")
print(test_df["label"].value_counts())

# Reset indices (needed for clean HuggingFace Dataset conversion)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [ ]:
# Load tokenizer and tokenize all splits
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Hyperparameter decision: max_length=512 (DistilBERT's max), not the
# notebook default of 256. In this dataset, due_diligence posts average
# ~585 words because the label is defined by cited evidence, and that
# evidence often sits in the back half of the post (after the framing/
# title). Truncating at 256 tokens would cut most posts off before the
# model ever sees the evidence that actually distinguishes due_diligence
# from hot_take. hot_take/reaction posts are short, so the longer window
# costs them nothing.
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

def make_dataset(df_split):
    ds = Dataset.from_pandas(
        df_split[["text", "label_id"]].rename(columns={"label_id": "labels"})
    )
    return ds.map(tokenize, batched=True)

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization complete")
print(f"Sample keys: {list(train_dataset[0].keys())}")

---
## Section 3: Fine-Tune Your Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on your training data.  
Training runs for 3 epochs and takes **5–15 minutes** on a T4 GPU.

> **Hyperparameter note:** The defaults below work well for datasets of 100–500 examples.  
> If you change any values, note what you changed and why in your README.

In [ ]:
# Load DistilBERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Output labels: {NUM_LABELS}")

In [ ]:
# ── FIX 1: Added macro F1 to compute_metrics ─────────────────────────────
# Original only computed accuracy. With hot_take at ~67% of the training
# set, a model can achieve 0.68 accuracy by predicting hot_take for *every*
# example — it never has to learn due_diligence or reaction at all.
# Adding macro F1 gives the Trainer a signal that actually rewards
# correct predictions on minority classes.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
    }


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
# num_train_epochs  — 5 (was 3). See FIX B note below.
# learning_rate     — 2e-5, standard for BERT-family fine-tuning.
# per_device_train_batch_size — 16.
#
# ── RUN 1 FAILURE: always predicted hot_take ──────────────────────────────
# hot_take was ~67% of training data. Plain cross-entropy + accuracy-based
# checkpoint selection meant the model could hit 0.68 accuracy by ignoring
# due_diligence and reaction entirely. It never had to learn those labels.
#
# ── RUN 2 FAILURE: always predicted due_diligence ─────────────────────────
# The inverse-frequency weight formula (len/n_classes/count) produced
# weights of ~2.0 for minority classes and ~0.5 for hot_take — a 4:1 ratio.
# That was too aggressive for 3 epochs on 114 training examples. The model
# overcorrected and swung to always predicting due_diligence.
#
# ── RUN 3 FIXES ───────────────────────────────────────────────────────────
# FIX A: Soft class weights via sqrt(balanced weights)
#   sklearn compute_class_weight('balanced') produces the right ratios but
#   the raw values remain too extreme for small data. Applying sqrt() compresses
#   a 4:1 ratio to a 2:1 ratio — enough pressure to prevent majority-class
#   collapse without triggering the opposite collapse.
#
# FIX B: Increase epochs 3 → 5
#   With ~19 minority training examples, 3 epochs gives only ~57 gradient
#   steps per minority class — not enough for the decision boundary to
#   stabilize. 5 epochs ≈ 95 steps. eval_strategy='epoch' +
#   load_best_model_at_end + macro F1 checkpoint selection guard against
#   overfitting from the additional epochs.
#
# FIX C: Label smoothing (label_smoothing_factor=0.1)
#   Replaces hard 0/1 targets with soft 0.033/0.933 values. Prevents the
#   model from becoming overconfident on the majority class early in training.
#   Works with the softer weights: weights redirect loss attention,
#   smoothing keeps confidence calibrated across all three classes.
# ─────────────────────────────────────────────────────────────────────────
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# FIX A: soft class weights
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=train_df["label_id"].values,
)
# sqrt() softens: e.g. 4:1 ratio becomes 2:1
soft_weights = np.sqrt(raw_weights)
# re-normalise so weights sum to NUM_LABELS (keeps overall loss scale stable)
soft_weights = soft_weights / soft_weights.mean()

print("Raw balanced weights:", {ID_TO_LABEL[i]: round(w, 2) for i, w in enumerate(raw_weights)})
print("Soft weights (sqrt + normalized):", {ID_TO_LABEL[i]: round(w, 2) for i, w in enumerate(soft_weights)})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor(soft_weights, dtype=torch.float).to(device)


class WeightedTrainer(Trainer):
    """Applies soft inverse-frequency class weights to cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=5,                      # FIX B: was 3
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    label_smoothing_factor=0.1,              # FIX C: new
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",        # from run 2: keeps macro F1 selection
    logging_steps=10,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting fine-tuning... (8–20 minutes on T4 GPU for 5 epochs)")
trainer.train()
print("\n✅ Fine-tuning complete")


---
## Section 4: Evaluate Fine-Tuned Model on Test Set

Runs inference on your locked test set and generates metrics and a confusion matrix.  
These numbers go directly into your evaluation report.

In [ ]:
# Run inference on the test set
print("Running inference on test set...")
ft_output = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

# Overall accuracy
ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
print(f"\n🎯 Fine-tuned model accuracy: {ft_accuracy:.3f}")

# Per-class metrics
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nPer-class metrics (fine-tuned model):")
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned Model — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("✅ Saved: confusion_matrix.png  →  commit this to your repo and include in README")

In [ ]:
# Print wrong predictions for your error analysis
# Review these carefully — pick 3 to analyze in depth in your README.

wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"Wrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for i, idx in enumerate(wrong_idx[:15]):
    text = test_df.iloc[idx]["text"]
    true_label = ID_TO_LABEL[ft_true_ids[idx]]
    pred_label = ID_TO_LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{i+1} ---")
    print(f"Text:      {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True:      {true_label}")
    print(f"Predicted: {pred_label}  (confidence: {confidence:.2f})")
    print()

---
## Section 5: Baseline Classifier (Groq)

Runs your zero-shot baseline using `llama-3.3-70b-versatile`.  
You need to write the classification prompt using your label definitions.

In [ ]:
from groq import Groq

# ── TODO: Add your Groq API key ───────────────────────────────────────────
# Recommended: use Colab Secrets so your key is never visible in the notebook.
#   1. Click the 🔑 icon in the left sidebar ("Secrets")
#   2. Add a secret named GROQ_API_KEY with your key as the value
#   3. Enable notebook access for the secret
#
# Then uncomment Option A below (and delete Option B).
#
# Option A — Colab Secrets (recommended):
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
#
# Option B — paste directly (do not commit to GitHub):
#GROQ_API_KEY = "your_groq_api_key_here"

assert GROQ_API_KEY, (
    "GROQ_API_KEY not set — add it in the Colab Secrets panel (\U0001f511, left "
    "sidebar) and enable notebook access for this notebook, or use Option B above."
)

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized")

In [ ]:
SYSTEM_PROMPT = """
You are classifying posts from r/wallstreetbets (WSB), an online stock trading community.
Assign each post to exactly one of the following three categories.

due_diligence: A research-backed argument that cites specific financial evidence (earnings, ratios, competitive dynamics, filings, historical comparisons) and uses that evidence to support a claim about a stock or the market. The post can be wrong and still count as due_diligence if it argues from facts rather than just asserting an opinion.
Example: "PYPL FCF yield is 9% at current price. Active accounts down 4% YoY but transactions per account up 11%. New management buying back $5B through 2025. Cheapest large-cap tech stock by FCF I can find right now. Long calls dated June."

hot_take: A bold, confident claim about a stock or the market with little to no specific supporting evidence. It may include a fact or two, but the post mainly asserts a conclusion rather than building an argument from evidence.
Example: "NVIDIA is the next Cisco. We're at the top, mark my words. $5000 puts."

reaction: An emotional response to a market event or a personal trading outcome. The post exists to express a feeling, not to inform or persuade — if you removed the emotion, there would be nothing left.
Example: "Lost 80% of my port on SPY puts today. My wife doesn't know yet. I'm done with options forever."

Respond with ONLY the label name. Do not explain your reasoning.

Valid labels:
due_diligence
hot_take
reaction
"""

print("Prompt length:", len(SYSTEM_PROMPT), "characters")

In [ ]:
def classify_with_groq(text):
    """Classify a single post. Returns a label string or None if unparseable."""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this post:\n\n{text}"},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = response.choices[0].message.content.strip().lower()
        # Match the model's output to a label. Check longest labels first so a
        # label that is a substring of another (e.g. "recommendation" vs.
        # "strong_recommendation") can't be matched by mistake.
        for label in sorted(LABEL_MAP, key=len, reverse=True):
            if raw == label or label in raw:
                return label
        return None  # model output didn't match any known label
    except Exception as e:
        print(f"API error: {e}")
        return None


# Run baseline on test set
print(f"Running baseline on {len(test_df)} examples...")
print("(May take a few minutes — 0.1s delay between requests to respect free-tier limits)\n")

baseline_preds = []
for i, (_, row) in enumerate(test_df.iterrows()):
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_df)} complete...")
    time.sleep(0.1)

none_count = baseline_preds.count(None)
if none_count > 0:
    print(f"\n⚠️  {none_count} responses could not be parsed.")
    print("Review your prompt — the model may not be outputting clean label names.")

In [ ]:
# Baseline metrics (exclude unparseable responses)
valid = [(p, t) for p, t in zip(baseline_preds, test_df["label_id"])
         if p is not None]
bl_pred_ids = [LABEL_MAP[p] for p, _ in valid]
bl_true_ids = [t for _, t in valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
print(f"🎯 Baseline accuracy: {bl_accuracy:.3f}  "
      f"(evaluated on {len(valid)}/{len(test_df)} parseable responses)")
print()
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("Per-class metrics (baseline):")
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))

---
## Section 6: Compare Results and Export

Side-by-side comparison of both models.  
Download the output files and commit them to your GitHub repo.

In [ ]:
print("=" * 50)
print("RESULTS COMPARISON")
print("=" * 50)
print(f"{'Model':<35} {'Accuracy':>8}")
print("-" * 45)
print(f"{'Zero-shot baseline (Groq)':<35} {bl_accuracy:>8.3f}")
print(f"{'Fine-tuned DistilBERT':<35} {ft_accuracy:>8.3f}")
print("-" * 45)
delta = ft_accuracy - bl_accuracy
direction = "improvement" if delta >= 0 else "regression"
print(f"\nFine-tuning {direction}: {abs(delta):.3f}")
print()
print("Use these numbers in your README evaluation report.")

In [ ]:
# ── FIX 3: Export includes macro F1 (not just accuracy) ──────────────────
# The original evaluation_results.json only stored accuracy, which is
# misleading on an imbalanced dataset (68% accuracy looked like an
# improvement over the 52% baseline, but macro F1 told the real story).
# ─────────────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score

ft_f1_macro    = f1_score(ft_true_ids,  ft_pred_ids,  average="macro", zero_division=0)
bl_f1_macro    = f1_score(bl_true_ids,  bl_pred_ids,  average="macro", zero_division=0)

results = {
    "baseline_accuracy":  round(bl_accuracy,  4),
    "finetuned_accuracy": round(ft_accuracy,  4),
    "baseline_f1_macro":  round(bl_f1_macro,  4),
    "finetuned_f1_macro": round(ft_f1_macro,  4),
    "improvement_accuracy": round(ft_accuracy - bl_accuracy, 4),
    "improvement_f1_macro": round(ft_f1_macro - bl_f1_macro, 4),
    "test_set_size": len(test_df),
    "label_map":     LABEL_MAP,
    "model":         MODEL_NAME,
}
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Files ready to download:")
print("   evaluation_results.json  — metrics for your README")
print("   confusion_matrix.png     — include in your README")
print()
print("Download via: Files panel (📁) on the left → right-click → Download")
